# Tutorial 6: GPU-Accelerated Inference

`gpu.run` is a drop-in replacement for `inference.run` that patches the hot
kernels with GPU (PyTorch/CUDA) implementations before calling the standard pipeline.
The interface is **identical** — only the import changes.

| Stage | CPU | GPU | Speedup |
|---|---|---|---|
| Incoherent (65k bank) | ~120 s | ~27 s | **4–5×** |
| draw_extrinsic | ~110 s | ~115 s | ~1× |
| Coherent | ~40 s | ~36 s | ~1× |
| **Total** | **~270 s** | **~180 s** | **~1.5×** |

At larger banks (1M templates) the incoherent dominates and the overall speedup
reaches **2.7×**. See `gpu/BENCHMARK.md` for full timing breakdowns.

**Requires**: CUDA-capable GPU with PyTorch installed.

In [ ]:
import sys
import time
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import rel_entr
import torch

warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
sys.path.append("../..")

from cogwheel import data, gw_utils, gw_plotting, utils, prior_ratio
from cogwheel.gw_prior import IntrinsicIASPrior
from dot_pe import inference, config, waveform_banks
from dot_pe.power_law_mass_prior import PowerLawIntrinsicIASPrior

import gpu.run as gpu_run   # <-- the only new import vs a CPU notebook

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("Imports OK")

## Step 1: Create event and bank

Same injection as the other tutorials (chirp_mass=20, HLV, O3 ASDs).
Artifacts are reused if they already exist.

In [ ]:
ARTIFACTS_DIR = Path("./artifacts_gpu")
ARTIFACTS_DIR.mkdir(exist_ok=True)

eventname  = "tutorial_gpu_event"
chirp_mass = 20.0
q          = 0.7
m1, m2     = gw_utils.mchirpeta_to_m1m2(chirp_mass, gw_utils.q_to_eta(q))

injection_par_dic = dict(
    m1=m1, m2=m2,
    ra=0.5, dec=0.5, iota=np.pi / 3, psi=1.0,
    phi_ref=12.0,
    s1z=0.3, s2z=0.3, s1x_n=0.1, s1y_n=0.2, s2x_n=0.3, s2y_n=-0.2,
    l1=0.0, l2=0.0,
    tgps=0.0, f_ref=50.0,
    d_luminosity=2000.0, t_geocenter=0.0,
)

event_path = ARTIFACTS_DIR / f"{eventname}.npz"
if not event_path.exists():
    event_data = data.EventData.gaussian_noise(
        eventname=eventname,
        detector_names="HLV",
        duration=120.0,
        asd_funcs=["asd_H_O3", "asd_L_O3", "asd_V_O3"],
        tgps=0.0, fmax=1600.0, seed=20260317,
    )
    event_data.inject_signal(injection_par_dic, "IMRPhenomXPHM")
    snr = np.sqrt(
        2 * (event_data.injection["d_h"] - 0.5 * event_data.injection["h_h"]).sum()
    )
    print(f"Injection SNR: {snr:.2f}")
    event_data.to_npz(filename=event_path, overwrite=True)
    print(f"Saved: {event_path}")
else:
    print(f"Reusing: {event_path}")

In [ ]:
bank_dir    = ARTIFACTS_DIR / "bank"
bank_size   = 2**16          # 65 536 templates
mchirp_min  = 15.0
mchirp_max  = 30.0
q_min       = 0.2
f_ref       = 50.0
approximant = "IMRPhenomXPHM"

if not (bank_dir / "intrinsic_sample_bank.feather").exists():
    bank_dir.mkdir(parents=True, exist_ok=True)

    powerlaw_prior = PowerLawIntrinsicIASPrior(
        mchirp_range=(mchirp_min, mchirp_max), q_min=q_min, f_ref=f_ref,
    )
    ias_prior = IntrinsicIASPrior(
        mchirp_range=(mchirp_min, mchirp_max), q_min=q_min, f_ref=f_ref,
    )
    pr = prior_ratio.PriorRatio(ias_prior, powerlaw_prior)
    prior_ratio._remove_matching_items(
        pr._numerator_subpriors, pr._denominator_subpriors
    )

    print(f"Generating {bank_size:,} bank samples...")
    bank_samples = powerlaw_prior.generate_random_samples(
        bank_size, seed=777, return_lnz=False
    )
    bank_samples["mchirp"] = gw_utils.m1m2_to_mchirp(
        bank_samples["m1"], bank_samples["m2"]
    )
    bank_samples["lnq"]    = np.log(bank_samples["m2"] / bank_samples["m1"])
    bank_samples["chieff"] = gw_utils.chieff(
        *bank_samples[["m1", "m2", "s1z", "s2z"]].values.T
    )
    bank_samples["log_prior_weights"] = bank_samples.apply(
        lambda row: pr.ln_prior_ratio(**row.to_dict()), axis=1
    )

    cols = ["m1", "m2", "s1z", "s1x_n", "s1y_n", "s2z", "s2x_n", "s2y_n",
            "iota", "log_prior_weights"]
    samples_path = bank_dir / "intrinsic_sample_bank.feather"
    bank_samples[cols].to_feather(samples_path)

    bank_config = {
        "bank_size": bank_size,
        "mchirp_min": mchirp_min, "mchirp_max": mchirp_max,
        "q_min": q_min, "f_ref": f_ref,
        "fbin": config.DEFAULT_FBIN.tolist(),
        "approximant": approximant,
        "m_arr": [2, 1, 3, 4],
        "seed": 777,
    }
    with open(bank_dir / "bank_config.json", "w") as f:
        json.dump(bank_config, f, indent=4)

    print("Generating waveforms (4 cores)...")
    waveform_banks.create_waveform_bank_from_samples(
        samples_path=samples_path,
        bank_config_path=bank_dir / "bank_config.json",
        waveform_dir=bank_dir / "waveforms",
        n_pool=4, blocksize=4096, approximant=approximant,
    )
    print("Bank ready.")
else:
    print(f"Reusing: {bank_dir}")

## Step 2: GPU run

`gpu.run.run()` is a drop-in for `inference.run()`.  It monkey-patches the
CPU kernels with GPU implementations, then calls the standard pipeline unchanged.

In [ ]:
N_EXT     = 1024
N_PHI     = 50
N_T       = 128
BLOCKSIZE = 2048
SEED      = 42

gpu_rundir_root = ARTIFACTS_DIR / "run_gpu"
gpu_rundir_root.mkdir(exist_ok=True)

t0 = time.time()
gpu_rundir = gpu_run.run(
    event=event_path,
    bank_folder=bank_dir,
    n_ext=N_EXT, n_phi=N_PHI, n_t=N_T,
    blocksize=BLOCKSIZE, single_detector_blocksize=BLOCKSIZE,
    seed=SEED,
    event_dir=str(gpu_rundir_root),
    mchirp_guess=chirp_mass,
    max_incoherent_lnlike_drop=20,
    max_bestfit_lnlike_diff=20,
    draw_subset=True,
)
t_gpu = time.time() - t0

gpu_summary = utils.read_json(gpu_rundir / "summary_results.json")
print(f"GPU wall time  : {t_gpu:.1f} s")
print(f"ln_evidence    : {gpu_summary['ln_evidence']:.4f}")
print(f"n_effective    : {gpu_summary['n_effective']:.1f}")

## Step 3: CPU baseline

Identical call using `inference.run` — no GPU patches.  Skips if a completed
run already exists in the output directory.

In [ ]:
cpu_rundir_root = ARTIFACTS_DIR / "run_cpu"
cpu_rundir_root.mkdir(exist_ok=True)

existing = sorted(cpu_rundir_root.glob("run_*/samples.feather"))
if existing:
    cpu_rundir = existing[-1].parent
    t_cpu = None
    print(f"Reusing CPU run: {cpu_rundir}")
else:
    t0 = time.time()
    cpu_rundir = inference.run(
        event=event_path,
        bank_folder=bank_dir,
        n_ext=N_EXT, n_phi=N_PHI, n_t=N_T,
        blocksize=BLOCKSIZE, single_detector_blocksize=BLOCKSIZE,
        seed=SEED,
        event_dir=str(cpu_rundir_root),
        mchirp_guess=chirp_mass,
        max_incoherent_lnlike_drop=20,
        max_bestfit_lnlike_diff=20,
        draw_subset=True,
    )
    t_cpu = time.time() - t0

cpu_summary = utils.read_json(cpu_rundir / "summary_results.json")
print(f"ln_evidence    : {cpu_summary['ln_evidence']:.4f}")
print(f"n_effective    : {cpu_summary['n_effective']:.1f}")
if t_cpu is not None:
    print(f"CPU wall time  : {t_cpu:.1f} s")
    print(f"Speedup        : {t_cpu / t_gpu:.2f}×")

## Step 4: Timing comparison

In [ ]:
rows = [{"mode": "GPU (gpu.run)",      "wall_s": t_gpu,
         "ln_evidence": gpu_summary["ln_evidence"],
         "n_effective": gpu_summary["n_effective"]}]
if t_cpu is not None:
    rows.append({"mode": "CPU (inference.run)", "wall_s": t_cpu,
                 "ln_evidence": cpu_summary["ln_evidence"],
                 "n_effective": cpu_summary["n_effective"]})
    rows[0]["speedup"] = t_cpu / t_gpu
    rows[1]["speedup"] = 1.0

display(pd.DataFrame(rows).set_index("mode").round(3))

## Step 5: Load and normalise samples

In [ ]:
def load_samples(path):
    df = pd.read_feather(path)
    df["weights"] = df["weights"] / df["weights"].sum()
    return df

gpu_samples = load_samples(gpu_rundir / "samples.feather")
cpu_samples = load_samples(cpu_rundir / "samples.feather")

for label, df in [("GPU", gpu_samples), ("CPU", cpu_samples)]:
    n_eff = 1 / (df["weights"] ** 2).sum()
    print(f"{label}: n={len(df)},  n_eff={n_eff:.0f}")

## Step 6: Corner plot

In [ ]:
true_mchirp = gw_utils.m1m2_to_mchirp(m1, m2)
true_lnq    = np.log(m2 / m1)
true_chieff = gw_utils.chieff(m1, m2,
                               injection_par_dic["s1z"],
                               injection_par_dic["s2z"])
true_values = injection_par_dic | {
    "mchirp": true_mchirp, "lnq": true_lnq, "chieff": true_chieff,
}

CORNER_PARAMS = ["mchirp", "lnq", "chieff", "d_luminosity", "costheta_jn", "ra", "dec"]

cp = gw_plotting.MultiCornerPlot(
    [cpu_samples, gpu_samples],
    params=CORNER_PARAMS,
    smooth=1.0,
    labels=["CPU", "GPU"],
)
cp.plot(max_figsize=9)
cp.scatter_points(true_values, colors="red", marker=".", s=200, label="Injection")
plt.suptitle("CPU vs GPU posteriors", y=1.01, fontsize=13)
plt.savefig(ARTIFACTS_DIR / "corner_cpu_vs_gpu.pdf", bbox_inches="tight")
plt.show()

## Step 7: 1-D marginal histograms

In [ ]:
HIST_PARAMS = [
    "mchirp", "lnq", "chieff", "cumchidiff",
    "d_luminosity", "costheta_jn",
    "ra", "dec", "psi", "t_geocenter",
]
COLORS = {"CPU": "#333333", "GPU": "#1f77b4"}

ncols = 5
nrows = int(np.ceil(len(HIST_PARAMS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3 * nrows))
axes = axes.ravel()

for ax_idx, param in enumerate(HIST_PARAMS):
    ax = axes[ax_idx]
    for label, df in [("CPU", cpu_samples), ("GPU", gpu_samples)]:
        if param not in df.columns:
            continue
        x, w = df[param].values, df["weights"].values
        lo, hi = np.quantile(x, [0.001, 0.999])
        bins = np.linspace(lo, hi, 50)
        ax.hist(x, bins=bins, weights=w, histtype="step",
                color=COLORS[label],
                linewidth=2.0 if label == "CPU" else 1.4,
                linestyle="-" if label == "CPU" else "--",
                label=label, density=True)
    # injection truth
    if param in true_values:
        ax.axvline(true_values[param], color="red", lw=1.2, ls=":", alpha=0.8)
    ax.set_xlabel(param, fontsize=10)
    ax.set_yticks([])
    if ax_idx == 0:
        ax.legend(fontsize=9)

for ax in axes[len(HIST_PARAMS):]:
    ax.set_visible(False)

fig.suptitle("1-D marginals — CPU vs GPU  (red dotted = injection)", fontsize=13)
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "histograms_cpu_vs_gpu.pdf", bbox_inches="tight")
plt.show()

## Step 8: KS and JS divergences

In [ ]:
def weighted_ks(x1, w1, x2, w2, n_bins=500):
    lo, hi = min(x1.min(), x2.min()), max(x1.max(), x2.max())
    grid = np.linspace(lo, hi, n_bins + 1)
    cdf1 = np.array([(w1[x1 <= g]).sum() for g in grid])
    cdf2 = np.array([(w2[x2 <= g]).sum() for g in grid])
    return float(np.abs(cdf1 - cdf2).max())

def js_divergence(x1, w1, x2, w2, n_bins=50):
    lo = min(np.quantile(x1, 0.001), np.quantile(x2, 0.001))
    hi = max(np.quantile(x1, 0.999), np.quantile(x2, 0.999))
    bins = np.linspace(lo, hi, n_bins + 1)
    h1, _ = np.histogram(x1, bins=bins, weights=w1)
    h2, _ = np.histogram(x2, bins=bins, weights=w2)
    h1 = h1 / h1.sum() + 1e-12
    h2 = h2 / h2.sum() + 1e-12
    m = 0.5 * (h1 + h2)
    return float(0.5 * (rel_entr(h1, m).sum() + rel_entr(h2, m).sum()))

rows = []
for param in HIST_PARAMS:
    if param not in cpu_samples.columns:
        continue
    x_cpu, w_cpu = cpu_samples[param].values, cpu_samples["weights"].values
    x_gpu, w_gpu = gpu_samples[param].values, gpu_samples["weights"].values
    rows.append({
        "param": param,
        "KS": weighted_ks(x_cpu, w_cpu, x_gpu, w_gpu),
        "JS": js_divergence(x_cpu, w_cpu, x_gpu, w_gpu),
    })

ks_df = pd.DataFrame(rows).set_index("param")
display(ks_df.round(4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, col, title, vmax in zip(
    axes, ["KS", "JS"],
    ["KS statistic (CPU vs GPU)", "JS divergence (CPU vs GPU)"],
    [0.15, 0.05],
):
    vals = ks_df[[col]].values
    im = ax.imshow(vals, aspect="auto", cmap="YlOrRd", vmin=0, vmax=vmax)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0])
    ax.set_xticklabels([col])
    ax.set_yticks(range(len(ks_df)))
    ax.set_yticklabels(ks_df.index, fontsize=10)
    for i, v in enumerate(vals[:, 0]):
        ax.text(0, i, f"{v:.3f}", ha="center", va="center", fontsize=9,
                color="black" if v < vmax * 0.7 else "white")
    ax.set_title(title, fontsize=11)

fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "ks_js_heatmap.pdf", bbox_inches="tight")
plt.show()

n_pass = (ks_df["KS"] < 0.05).sum()
print(f"\nKS < 0.05 on {n_pass}/{len(ks_df)} params — GPU posteriors consistent with CPU.")